[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter13_optimizers_sgd_momentum_and_adam/notebook.ipynb)

# Chapter 13: Optimizers: SGD, Momentum and Adam

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Every step you have taken downhill so far has used the plainest possible rule, which is to move each weight opposite to its slope by a fixed fraction. That rule is Cauchy's from Chapter 6, and it has a weakness that shows up badly in real networks.

In 1964 a Soviet mathematician named Boris Polyak suggested a fix so simple it is almost a joke. Give the ball some weight. Instead of deciding each step purely from the slope under your feet, keep a running sense of the direction you have been travelling and let it carry you. Polyak called it the heavy ball method, and it is now called **momentum**.

Nearly fifty years later a second idea arrived from a different direction. If some weights sit in steep parts of the landscape and others in flat parts, why should they all take steps of the same size? Methods that give every weight its own step size appeared as AdaGrad in 2011 and RMSProp, which Geoffrey Hinton introduced in a lecture rather than a paper. Then in late 2014 Diederik Kingma and Jimmy Ba published a method that simply combined the two, momentum and per weight step sizes together, and called it **Adam**. It was presented at a conference in San Diego in 2015 and became the default almost everywhere.

This chapter replaces the plain step with those methods. Before you read on, try to guess the answers to these four questions.

1. **What shape of valley would make a plain downhill step behave badly?**
2. **What would giving the step some inertia actually do?**
3. **Why might two different weights want different step sizes?**
4. **If one optimizer is best, why would anyone still use the others?**

## Plain SGD and Its Problems

The trouble comes from a shape called a **ravine**, which is a valley that is steep across and nearly flat along its length. Picture a long narrow gorge. The way to the bottom is to walk down the length of it, but the steepest direction from where you stand is across it, towards the far wall.

The next cell builds exactly such a loss over two weights, steep in one and shallow in the other, with its lowest point at the origin.

In [ ]:
def loss(x, y):
    return 0.1 * x ** 2 + 2 * y ** 2

def slopes(x, y):
    return 0.2 * x, 4 * y

The $y$ direction is twenty times steeper than the $x$ direction. Now walk downhill from a starting point well up the gorge, taking twenty plain steps.

In [ ]:
def descend(learning_rate, beta=0.0, steps=20):
    x, y = -9.0, 2.0
    vx = vy = 0.0
    path = [(x, y)]
    for _ in range(steps):
        gx, gy = slopes(x, y)
        vx = beta * vx + gx
        vy = beta * vy + gy
        x -= learning_rate * vx
        y -= learning_rate * vy
        path.append((x, y))
    return path

With `beta` left at zero that function is precisely Chapter 6's gradient descent. The next cell runs it and shows what the two coordinates do.

In [ ]:
plain = descend(0.4)
print([round(y, 2) for x, y in plain[:6]])
print([round(x, 2) for x, y in plain[:6]])

Read those two lines together, because they are the whole problem. The steep coordinate flings itself back and forth across the valley, overshooting the bottom every time and changing sign at every step. Meanwhile the shallow coordinate, the one that actually leads somewhere, creeps from $-9$ to $-6.45$ in five steps. After twenty steps it has still not arrived.

In [ ]:
print(round(plain[-1][0], 3), round(loss(*plain[-1]), 4))

Most of the effort went into bouncing rather than travelling. You cannot fix it by turning the learning rate down, because that slows the useful direction just as much, and you cannot turn it up, because the bouncing direction is already on the edge of flying apart. Chapter 6 warned that one learning rate has to serve every weight at once, and this is what that costs.

## Momentum

Polyak's fix is in the two lines you already saw. Rather than stepping by the slope, keep a running total called the **velocity**, add each new slope to it, and step by that instead. The `beta` setting says how much of the previous velocity survives, so 0.5 keeps half of it and 0 keeps none, which is plain descent again.

Think of a ball with real weight rolling down the gorge. Each time it is pushed towards a wall, the push is partly cancelled by the push it got from the opposite wall a moment earlier, because those two cancel in the running total. The pushes along the valley all point the same way, so they add up instead. The result is that the bouncing dies down while the useful direction speeds up.

In [ ]:
moving = descend(0.4, beta=0.5)
print(round(moving[-1][0], 3), round(loss(*moving[-1]), 4))

The same learning rate, the same twenty steps, and the same starting point. Plain descent ended at $x=-1.698$ with a loss of 0.2884, while momentum reached $x=-0.113$ with a loss of 0.0013, more than two hundred times lower. Figure 13.1 draws both paths down the gorge.

> 🖼️ **Figure 13.1:** The same ravine walked two ways. Plain descent zigzags across the steep direction and barely advances, while momentum cancels the zigzag and sails along the valley floor.

One honest caution. The value 0.5 was chosen for this gorge at this learning rate, and the common default in real work is 0.9, which keeps far more of the previous velocity. On this particular problem 0.9 overshoots badly at a learning rate of 0.4 and needs a much smaller one to behave. **Momentum and the learning rate are tuned together, because momentum effectively lengthens the step, so raising one usually means lowering the other.**

PyTorch needs no new optimizer for this, since `SGD` takes momentum as an argument.

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(4, 3)
opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
group = opt.param_groups[0]
print(group["lr"], group["momentum"])

Those are the only two settings here that matter in everyday use, and this is also the answer to one of Chapter 6's promises, since momentum is the second of the four reasons that training gets out of shallow dips. A run with inertia rolls through a small dip the way a ball rolls through a pothole.

## Adam and the Adaptive Family

Momentum fixes the direction of the step. The second family of methods fixes its size, separately for every single weight.

The reasoning is the one the ravine already showed. In that gorge the steep weight needed small steps and the shallow weight needed large ones, and no single learning rate could give both. If instead each weight kept a record of how large its own slopes have recently been, it could scale its own step accordingly, taking small steps where slopes are big and large steps where they are small.

AdaGrad did this first, in 2011, by dividing each weight's step by the total size of all its past slopes. That works but has a flaw, since the total only ever grows, so every step eventually shrinks towards nothing and learning stalls. RMSProp fixed it by keeping a decaying average instead of a total, so old slopes fade out of the record.

Adam is the two ideas bolted together. It keeps a running average of recent slopes, which is momentum, and a running average of recent squared slopes, which is the per weight step size, and divides one by the square root of the other. The result is that each weight gets its own adaptive step in the direction momentum is already carrying it.

Table 13.1 sets out the family without any formulas.

**Table 13.1: How the common optimizers relate. Each one adds a single idea to the one before it, and Adam is the combination of the two main branches.**

| Optimizer | Its idea | Arrived |
|---|---|---|
| SGD | Step opposite the slope | Cauchy 1847, Robbins and Monro 1951 |
| SGD with momentum | Keep a running direction | Polyak 1964 |
| AdaGrad | A step size per weight, from all past slopes | 2011 |
| RMSProp | The same, but old slopes fade | Hinton, around 2012 |
| Adam | Momentum and fading step sizes together | Kingma and Ba, 2014 to 2015 |

Now put all three to work on the flowers from Chapter 11, in the batches from Chapter 12.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

The data preparation is the same as Chapter 11's, split and scaled from the training half alone. Keeping it identical is the point of the exercise. The only thing that changes between the three runs below is the optimizer, so any difference in the curves belongs to the optimizer and to nothing else.

In [ ]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3,
    random_state=7, stratify=iris.target)
mean, std = X_train.mean(axis=0), X_train.std(axis=0)
Xtr = torch.tensor((X_train - mean) / std, dtype=torch.float32)
Xte = torch.tensor((X_test - mean) / std, dtype=torch.float32)
ytr, yte = torch.tensor(y_train), torch.tensor(y_test)
train_ds = TensorDataset(Xtr, ytr)
loss_fn = nn.CrossEntropyLoss()

The training function is Chapter 12's, with the choice of optimizer handed in so the three runs are otherwise identical.

In [ ]:
def run(make_optimizer, epochs=30):
    torch.manual_seed(7)
    net = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
    optimizer = make_optimizer(net)
    loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    torch.manual_seed(7)
    for epoch in range(epochs):
        for xb, yb in loader:
            l = loss_fn(net(xb), yb)
            optimizer.zero_grad()
            l.backward()
            optimizer.step()
    with torch.no_grad():
        return (loss_fn(net(Xtr), ytr).item(),
                (net(Xte).argmax(1) == yte).float().mean().item())

In [ ]:
choices = [
    ("SGD", lambda n: torch.optim.SGD(n.parameters(), lr=0.1)),
    ("momentum", lambda n: torch.optim.SGD(n.parameters(), lr=0.1,
                                           momentum=0.9)),
    ("Adam", lambda n: torch.optim.Adam(n.parameters(), lr=0.01))]

for name, make in choices:
    final, acc = run(make)
    print(name, round(final, 4), round(acc, 3))

Both improvements beat plain SGD and both reach the same test accuracy, missing one flower where SGD missed two. Figure 13.2 shows the three runs epoch by epoch.

> 🖼️ **Figure 13.2:** The same network, data and batches under three optimizers. Both momentum and Adam pull away from plain SGD within the first few epochs.

Notice that momentum ends slightly lower than Adam here. That is worth saying plainly, because Adam has a reputation for being the best choice everywhere and on a problem this small it is not. Adam's advantages show up on large, messy, high dimensional problems, which is where almost all modern work happens, and not on 105 flowers.

## Which Optimizer to Reach For

The practical answer is short. Start with Adam, because it works acceptably almost everywhere with very little tuning, and that matters more than squeezing out the last fraction of loss while you are still finding out whether an idea works at all. Its usual starting rate is 0.001, and the next cell tries three values on the flowers.

In [ ]:
for lr in [0.1, 0.01, 0.001]:
    final, acc = run(lambda n: torch.optim.Adam(n.parameters(), lr=lr))
    print(lr, round(final, 4), round(acc, 3))

That is not the answer the recommended default would lead you to expect, and it is worth understanding rather than explaining away. The usual 0.001 came last here, with a loss of 0.3211 and the worst accuracy of the three. It was not a bad choice, it was a slow one, and thirty epochs on 105 flowers simply did not give it enough steps to arrive. The next cell gives all three ten times as long.

In [ ]:
for lr in [0.1, 0.01, 0.001]:
    final, acc = run(lambda n: torch.optim.Adam(n.parameters(), lr=lr),
                     epochs=300)
    print(lr, round(final, 4), round(acc, 3))

With room to run, 0.001 reaches 0.0385 and the same accuracy as the rest, while 0.01 becomes the clear winner at 0.0027. **A learning rate is not right or wrong on its own, it is right or wrong for the number of steps you are willing to take, so a rate that looks too small may only be too slow.**

That is why 0.001 remains the sensible default in real work even though it lost here. Real problems run for thousands of steps rather than a few hundred, and on those a cautious rate that keeps improving beats an ambitious one that stalls early. On 105 flowers there is no room for that caution to pay off. It also shows that adapting a step size per weight, which is Adam's whole idea, is not the same as needing no learning rate at all, and Chapter 14 is entirely about finding that number properly instead of trying three.

**Table 13.2: A practical guide. Adam is the sensible default, and tuned SGD with momentum is the thing that sometimes beats it.**

| Situation | Reach for | Starting learning rate |
|---|---|---|
| Anything new, or you are unsure | Adam | 0.001 |
| Images, where tuned SGD often wins | SGD with momentum 0.9 | 0.01 to 0.1 |
| Reproducing a paper | Whatever it used | Whatever it used |
| Teaching or debugging by hand | Plain SGD | Whatever Chapter 14 finds |

**Use Adam at a learning rate of 0.001 unless you have a reason not to, remember that it is not magic and still needs that rate chosen, and know that a carefully tuned SGD with momentum can still beat it.**

That is also the end of a thread. Chapter 6 listed four reasons that gradient descent manages to find good valleys despite a landscape full of dips, and two of them are now delivered, since Chapter 12 gave the batch noise and this chapter gave the momentum. The remaining two, careful starting weights and the training recipe that holds it all together, arrive in Chapter 16.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Find momentum's limit.** Run `descend(0.4, beta=b)` for `b` of 0.7, 0.8 and 0.9, print the final loss for each, and find where too much inertia starts to hurt.
2. **Give plain SGD a fair fight.** Try `descend` with no momentum at learning rates of 0.1, 0.2 and 0.45, and show that no single value fixes both the bouncing and the crawling.
3. **Match Adam's budget.** Train with Adam at 0.001 for 90 epochs rather than 30 and see whether it catches the momentum run's 0.0272.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What is a ravine, and why does plain descent handle it badly?**
2. **How does momentum cancel the bouncing but not the useful direction?**
3. **What problem did AdaGrad have that RMSProp fixed?**
4. **What two ideas does Adam combine?**
5. **Why is Adam's usual learning rate so much smaller than SGD's?**

### Answers

1. A valley that is steep across and nearly flat along. The steepest direction points across it, so plain steps bounce between the walls while barely advancing along it.
2. Because the pushes towards opposite walls alternate in sign and cancel in the running total, while the pushes along the valley all point the same way and add up.
3. AdaGrad divided by the total of all past slopes, which only grows, so steps shrank towards nothing. RMSProp used a decaying average so old slopes fade.
4. Momentum, a running average of recent slopes, and a per weight step size from a running average of recent squared slopes.
5. Because Adam already scales each weight's step by its own recent slopes, so the overall rate is multiplying an already adjusted step rather than a raw slope.

## What Comes Next

You can now explain why plain descent struggles in a ravine, use momentum to cure it, describe where Adam came from and choose an optimizer with reasons rather than habit.

- **Chapter 14, Learning Rates and Schedules,** finds the number that every optimizer in this chapter still needed, and changes it as training goes on.
- **Chapter 15, Overfitting and Regularization,** deals with the curve in Chapter 11's table that falls on training data and rises on held out data.

Chapter 14 begins with a trick that finds a good learning rate in a single short run, rather than by guessing and waiting.